In [1]:
!pip install -q optuna

In [2]:
!pip install -q catboost

In [3]:
!pip install -q xgboost 

# Explore the Data

In [4]:
import pandas as pd

train = pd.read_csv("/kaggle/input/datasets/tolbaadel/dataset/train_test.csv")
validation = pd.read_csv("/kaggle/input/datasets/tolbaadel/dataset/validation.csv")
december = pd.read_csv("/kaggle/input/datasets/tolbaadel/dataset/december_chart_inputs.csv")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("December:", december.shape)

print("\nTrain columns:")
print(train.columns.tolist())

print("\nValidation columns:")
print(validation.columns.tolist())

print("\nDecember columns:")
print(december.columns.tolist())

Train: (48000, 14)
Validation: (12000, 13)
December: (31, 7)

Train columns:
['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate']

Validation columns:
['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal']

December columns:
['pickup', 'delivery', 'distance', 'equipment', 'weight', 'date', 'predicted_rate']


In [5]:
print("=== TRAIN INFO ===")
train.info()

print("\n=== TRAIN MISSING VALUES ===")
print(train.isna().sum())

print("\n=== TRAIN DUPLICATES ===")
print(train.duplicated().sum())

print("\n=== TARGET SUMMARY ===")
print(train["posted_rate"].describe())

print("\n=== DATE RANGE ===")
print(train["date"].min(), "->", train["date"].max())

print("\n=== EQUIPMENT ===")
print(train["equipment"].value_counts(dropna=False))

print("\n=== UNIQUE PICKUP LOCATIONS ===")
print(train["pickup"].nunique())

print("\n=== UNIQUE DELIVERY LOCATIONS ===")
print(train["delivery"].nunique())

=== TRAIN INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48000 entries, 0 to 47999
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   load_id       48000 non-null  object 
 1   pickup        48000 non-null  object 
 2   delivery      48000 non-null  object 
 3   pickup_lat    48000 non-null  float64
 4   pickup_lon    48000 non-null  float64
 5   delivery_lat  48000 non-null  float64
 6   delivery_lon  48000 non-null  float64
 7   distance      48000 non-null  float64
 8   equipment     48000 non-null  object 
 9   weight        47700 non-null  float64
 10  date          48000 non-null  object 
 11  market_index  47626 non-null  float64
 12  quote_signal  48000 non-null  float64
 13  posted_rate   48000 non-null  float64
dtypes: float64(9), object(5)
memory usage: 5.1+ MB

=== TRAIN MISSING VALUES ===
load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
de

In [6]:
print("=== FIRST 5 ROWS ===")
display(train.head())

print("=== NUMERIC SUMMARY ===")
display(train.describe(include="all").T)

=== FIRST 5 ROWS ===


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28


=== NUMERIC SUMMARY ===


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
load_id,48000,48000,TR-047961,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pickup,48000,64,Oklahoma City,1242,NaN,NaN,NaN,NaN,NaN,NaN,NaN
delivery,48000,64,Lexington,1197,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pickup_lat,48000.0,NaN,NaN,NaN,35.647545,4.315285,28.35765,31.98691,35.29479,39.41104,44.30296
pickup_lon,48000.0,NaN,NaN,NaN,-90.928964,13.482431,-121.69849,-98.40059,-88.08915,-83.28506,-69.5
delivery_lat,48000.0,NaN,NaN,NaN,35.641175,4.317199,28.35765,31.98691,35.29479,39.41104,44.30296
delivery_lon,48000.0,NaN,NaN,NaN,-90.85731,13.476589,-121.69849,-98.40059,-87.52871,-83.28506,-69.5
distance,48000.0,NaN,NaN,NaN,1135.856654,728.564416,70.0,550.4,953.3,1645.525,3439.8
equipment,48000,3,Dry Van,27202,NaN,NaN,NaN,NaN,NaN,NaN,NaN
weight,47700.0,NaN,NaN,NaN,31028.844004,9391.44062,-47500.0,25800.0,31436.5,37018.0,47500.0


## investigate the data quality

### A. Investigate negative weights

In [7]:
print("Negative weights:")
print((train["weight"] < 0).sum())

print("\nZero weights:")
print((train["weight"] == 0).sum())

print("\nWeight distribution:")
display(
    train["weight"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
)

print("\nRows with negative weight:")
display(
    train.loc[train["weight"] < 0, 
              ["pickup", "delivery", "distance", "equipment",
               "weight", "date", "market_index",
               "quote_signal", "posted_rate"]]
    .head(10)
)

Negative weights:
292

Zero weights:
0

Weight distribution:


count    47700.000000
mean     31028.844004
std       9391.440620
min     -47500.000000
1%        9801.960000
5%       17490.950000
25%      25800.000000
50%      31436.500000
75%      37018.000000
95%      44934.200000
99%      47500.000000
max      47500.000000
Name: weight, dtype: float64


Rows with negative weight:


,pickup,delivery,distance,equipment,weight,date,market_index,quote_signal,posted_rate
68,Amarillo,Dayton,1334.2,Reefer,-36559.0,2025-01-01,0.96102,2.11777,2831.27
204,Fresno,Tucson,496.2,Dry Van,-26670.0,2025-01-02,0.96903,2.13155,1046.92
206,Columbia,Raleigh,411.3,Reefer,-20003.0,2025-01-02,0.94800,2.31682,959.15
225,Greensboro,Lubbock,1317.4,Reefer,-23981.0,2025-01-02,1.00820,2.13918,2836.00
376,Madison,Providence,1331.1,Dry Van,-41397.0,2025-01-03,0.94787,1.91867,2522.54
446,Albuquerque,Albany,2438.3,Reefer,-37215.0,2025-01-03,0.96575,1.96959,4841.57
495,Montgomery,Dayton,715.9,Dry Van,-31214.0,2025-01-04,0.85153,2.03947,1473.78
641,Tulsa,Indianapolis,582.4,Dry Van,-41023.0,2025-01-05,0.81988,2.22721,1264.33
806,Cincinnati,Buffalo,676.1,Dry Van,-28320.0,2025-01-06,0.79158,1.95680,1307.76
1093,Cincinnati,Albany,1058.5,Dry Van,-25153.0,2025-01-07,0.89891,1.81081,1907.83


### Check the target outliers

In [8]:
print("Top 15 posted rates:")
display(
    train[
        ["pickup", "delivery", "distance", "equipment",
         "weight", "date", "market_index",
         "quote_signal", "posted_rate"]
    ]
    .sort_values("posted_rate", ascending=False)
    .head(15)
)

Top 15 posted rates:


,pickup,delivery,distance,equipment,weight,date,market_index,quote_signal,posted_rate
12184,Bakersfield,Hartford,2829.8,Dry Van,33153.0,2025-03-19,1.16556,1.83188,25533.00
37764,Phoenix,Syracuse,2810.9,Dry Van,33674.0,2025-08-27,1.04877,2.16284,24294.98
1466,Los Angeles,Syracuse,2786.0,Reefer,30322.0,2025-01-10,0.98678,1.85370,24140.21
17371,Reno,Philadelphia,2777.6,Reefer,33158.0,2025-04-20,1.15070,2.15601,23662.71
26235,Charleston,Phoenix,2552.5,Dry Van,25665.0,2025-06-15,1.20224,1.81289,23580.42
28219,Tucson,Raleigh,2423.3,Dry Van,30972.0,2025-06-27,1.32894,1.97112,22755.66
3353,Phoenix,Charleston,2527.9,Reefer,36840.0,2025-01-22,1.03241,2.07342,22534.65
40171,Montgomery,Fresno,2283.4,Dry Van,32447.0,2025-09-11,0.98502,1.93645,20361.89
40096,Baltimore,Oklahoma City,1760.3,Reefer,29521.0,2025-09-11,1.00464,2.28753,20132.37
47298,Boston,Bakersfield,2979.1,Reefer,30585.0,2025-10-27,0.86501,2.09472,19110.30


In [9]:
print("Rates above 5,000:", (train["posted_rate"] > 5000).sum())
print("Rates above 10,000:", (train["posted_rate"] > 10000).sum())
print("Rates above 20,000:", (train["posted_rate"] > 20000).sum())

Rates above 5,000: 2265
Rates above 10,000: 103
Rates above 20,000: 9


## understand the temporal structure

In [10]:
train["date"] = pd.to_datetime(train["date"])

daily_stats = (
    train.groupby("date")["posted_rate"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
)

display(daily_stats.head())
display(daily_stats.tail())

,date,count,mean,median,std
0,2025-01-01,145,2182.959586,1991.870,1193.423009
1,2025-01-02,168,2223.485595,1885.395,1349.279535
2,2025-01-03,156,2331.391154,1892.985,1412.795887
3,2025-01-04,156,2334.728782,1967.150,1534.821331
4,2025-01-05,154,2041.046688,1626.405,1278.917472


,date,count,mean,median,std
299,2025-10-27,148,2397.204122,1997.100,1929.581592
300,2025-10-28,131,2420.863511,2095.300,1322.776876
301,2025-10-29,164,2418.749085,2017.380,1802.173228
302,2025-10-30,177,2433.295593,1898.300,1586.881219
303,2025-10-31,176,2245.691023,1906.235,1417.564628


In [11]:
monthly_stats = (
    train.assign(month=train["date"].dt.to_period("M"))
    .groupby("month")["posted_rate"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
)

display(monthly_stats)

,month,count,mean,median,std
0,2025-01,4918,2255.967048,1915.200,1454.274952
1,2025-02,4337,2273.804801,1994.250,1317.077105
2,2025-03,5036,2372.268092,2022.920,1462.531615
3,2025-04,4819,2372.162308,2044.240,1463.102457
4,2025-05,4913,2421.776342,2065.510,1486.071436
5,2025-06,4783,2497.030115,2120.220,1606.707302
6,2025-07,4912,2415.162030,2059.145,1504.943296
7,2025-08,4759,2338.407661,2015.730,1474.979672
8,2025-09,4670,2406.374013,2057.130,1523.483983
9,2025-10,4853,2379.051374,2035.900,1528.714038


## check train vs validation

In [13]:
print("=== CATEGORICAL COVERAGE ===")

for col in ["pickup", "delivery", "equipment"]:
    train_values = set(train[col].dropna().unique())
    val_values = set(validation[col].dropna().unique())

    print(f"\n{col}")
    print("Train unique:", len(train_values))
    print("Validation unique:", len(val_values))
    print("Only in validation:", sorted(val_values - train_values))
    print("Only in train:", sorted(train_values - val_values))

=== CATEGORICAL COVERAGE ===

pickup
Train unique: 64
Validation unique: 72
Only in validation: ['Allentown', 'Charlotte', 'Chicago', 'Jackson', 'Knoxville', 'Laredo', 'Norfolk', 'San Diego']
Only in train: []

delivery
Train unique: 64
Validation unique: 72
Only in validation: ['Allentown', 'Charlotte', 'Chicago', 'Jackson', 'Knoxville', 'Laredo', 'Norfolk', 'San Diego']
Only in train: []

equipment
Train unique: 3
Validation unique: 3
Only in validation: []
Only in train: []


In [14]:
print("=== NUMERIC RANGES ===")

numeric_cols = [
    "pickup_lat", "pickup_lon",
    "delivery_lat", "delivery_lon",
    "distance", "weight",
    "market_index", "quote_signal"
]

comparison = pd.DataFrame({
    "train_min": train[numeric_cols].min(),
    "train_max": train[numeric_cols].max(),
    "validation_min": validation[numeric_cols].min(),
    "validation_max": validation[numeric_cols].max()
})

display(comparison)

=== NUMERIC RANGES ===


,train_min,train_max,validation_min,validation_max
pickup_lat,28.35765,44.30296,25.50000,44.30296
pickup_lon,-121.69849,-69.50000,-121.69849,-69.50000
delivery_lat,28.35765,44.30296,25.50000,44.30296
delivery_lon,-121.69849,-69.50000,-121.69849,-69.50000
distance,70.00000,3439.80000,70.00000,3325.70000
weight,-47500.00000,47500.00000,-47500.00000,47500.00000
market_index,0.67639,1.46778,0.72361,1.09895
quote_signal,0.69228,3.61035,1.23476,2.89635


# Prepare the Data

In [78]:
import numpy as np


def prepare_features(df):
    """
    Prepare freight-load features for modeling.

    Args:
        df: Raw freight-load DataFrame.

    Returns:
        DataFrame containing cleaned and engineered model features.
    """
    df = df.copy()

    # Parse date and create calendar features
    df["date"] = pd.to_datetime(df["date"])
    df["month"] = df["date"].dt.month
    df["day_of_month"] = df["date"].dt.day
    df["day_of_week"] = df["date"].dt.dayofweek

    # Negative freight weights are physically invalid, treat them as missing
    df.loc[df["weight"] < 0, "weight"] = np.nan

    # Remove identifier, target, raw date and useless columns
    df = df.drop(
        columns=["load_id", "posted_rate", "date", 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'market_index', 'quote_signal'],
        errors="ignore"
    )

    return df

## chronological split

In [79]:
train["date"] = pd.to_datetime(train["date"])

# September and earlier for training, October for validation
dev = train[train["date"] < "2025-10-01"].copy()
holdout = train[train["date"] >= "2025-10-01"].copy()

print("Development:", dev.shape)
print("Holdout:", holdout.shape)
print("Development:", dev["date"].min(), "->", dev["date"].max())
print("Holdout:", holdout["date"].min(), "->", holdout["date"].max())

Development: (43147, 14)
Holdout: (4853, 14)
Development: 2025-01-01 00:00:00 -> 2025-09-30 00:00:00
Holdout: 2025-10-01 00:00:00 -> 2025-10-31 00:00:00


## prepare X/y

In [80]:
target = "posted_rate"

X_dev = prepare_features(dev)
y_dev = dev[target]
y_dev_log = np.log1p(y_dev)

X_holdout = prepare_features(holdout)
y_holdout = holdout[target]
y_holdout_log = np.log1p(y_holdout)

print(y_dev_log.describe())

categorical_features = [
    X_dev.columns.get_loc("pickup"),
    X_dev.columns.get_loc("delivery"),
    X_dev.columns.get_loc("equipment"),
]

print(X_dev.shape)
print(X_dev.columns.tolist())

count    43147.000000
mean         7.573844
std          0.664823
min          4.064229
25%          7.135066
50%          7.616136
75%          8.111907
max         10.147766
Name: posted_rate, dtype: float64
(43147, 8)
['pickup', 'delivery', 'distance', 'equipment', 'weight', 'month', 'day_of_month', 'day_of_week']


In [86]:
X_dev.head()

,pickup,delivery,distance,equipment,weight,month,day_of_month,day_of_week
0,Richmond,Baltimore,274.3,Dry Van,30658.0,1,1,2
1,Richmond,Philadelphia,280.5,Reefer,17555.0,1,1,2
2,Philadelphia,Green Bay,967.8,Dry Van,31721.0,1,1,2
3,Hartford,Atlanta,965.4,Dry Van,32333.0,1,1,2
4,Dallas,Nashville,541.9,Reefer,35183.0,1,1,2


In [82]:
X_dev.tail()

,pickup,delivery,distance,equipment,weight,month,day_of_month,day_of_week
43142,Philadelphia,Green Bay,970.6,Dry Van,47500.0,9,30,1
43143,Albuquerque,Oklahoma City,803.6,Reefer,18014.0,9,30,1
43144,Kansas City,Tucson,1623.9,Reefer,36536.0,9,30,1
43145,Lexington,Dallas,744.2,Dry Van,32879.0,9,30,1
43146,Oklahoma City,Indianapolis,860.4,Dry Van,35820.0,9,30,1


# Run CatBoost Model

In [83]:
import optuna
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def objective(trial):
    """
    Train one CatBoost configuration selected by Optuna
    and return its holdout RMSE.

    Args:
        trial: Optuna trial containing sampled hyperparameters.

    Returns:
        Holdout RMSE for the sampled configuration.
    """

    model = CatBoostRegressor(
        iterations=trial.suggest_int("iterations", 500, 1500),
        learning_rate=trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        depth=trial.suggest_int("depth", 4, 10),
        l2_leaf_reg=trial.suggest_float(
            "l2_leaf_reg", 1, 20, log=True
        ),
        random_strength=trial.suggest_float(
            "random_strength", 0, 5
        ),
        bagging_temperature=trial.suggest_float(
            "bagging_temperature", 0, 5
        ),
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42,
        verbose=False
    )

    model.fit(
        X_dev,
        y_dev_log,
        cat_features=categorical_features,
        eval_set=(X_holdout, y_holdout_log),
        early_stopping_rounds=100,
        verbose=False
    )

    # Predictions are currently in log-space
    pred_log = model.predict(X_holdout)

    # Convert back to the original dollar scale
    pred_raw = np.expm1(pred_log)

    return np.sqrt(mean_squared_error(y_holdout, pred_raw))

In [84]:
study_cat = optuna.create_study(
    direction="minimize",
    study_name="catboost_freight_rate"
)

study_cat.optimize(objective, n_trials=30)

print("Best RMSE:", study_cat.best_value)
print("Best parameters:")
print(study_cat.best_params)

[I 2026-09-13 09:50:42,864] A new study created in memory with name: catboost_freight_rate
[I 2026-09-13 09:50:50,847] Trial 0 finished with value: 648.047534871379 and parameters: {'iterations': 739, 'learning_rate': 0.08693596589403167, 'depth': 6, 'l2_leaf_reg': 11.49320435606681, 'random_strength': 2.3530671086089363, 'bagging_temperature': 4.390465941480789}. Best is trial 0 with value: 648.047534871379.
[I 2026-09-13 09:51:05,045] Trial 1 finished with value: 651.8899840269155 and parameters: {'iterations': 547, 'learning_rate': 0.010529994408141964, 'depth': 9, 'l2_leaf_reg': 13.39079542846963, 'random_strength': 3.383867888158521, 'bagging_temperature': 0.04049070265308263}. Best is trial 0 with value: 648.047534871379.
[I 2026-09-13 09:51:27,480] Trial 2 finished with value: 648.1946553755012 and parameters: {'iterations': 1387, 'learning_rate': 0.020430968697576594, 'depth': 8, 'l2_leaf_reg': 5.519532355650333, 'random_strength': 1.742316543234335, 'bagging_temperature': 4.92

Best RMSE: 647.1947357996778
Best parameters:
{'iterations': 686, 'learning_rate': 0.10486604635909791, 'depth': 7, 'l2_leaf_reg': 6.988990741530071, 'random_strength': 1.1828053962394336, 'bagging_temperature': 0.7804249353008069}


## evaluation

In [85]:
best_cat = CatBoostRegressor(
    **study_cat.best_params,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=100
)

best_cat.fit(
    X_dev,
    y_dev_log,
    cat_features=categorical_features,
    eval_set=(X_holdout, y_holdout_log),
    early_stopping_rounds=100
)

# Predict in log-space
pred_log = best_cat.predict(X_holdout)

# Convert predictions back to original freight-rate scale
pred_cat = np.expm1(pred_log)


mae_cat = mean_absolute_error(y_holdout, pred_cat)
rmse_cat = np.sqrt(mean_squared_error(y_holdout, pred_cat))
r2_cat = r2_score(y_holdout, pred_cat)

print(f"MAE:  {mae_cat:,.2f}")
print(f"RMSE: {rmse_cat:,.2f}")
print(f"R²:   {r2_cat:.4f}")

0:	learn: 0.6027155	test: 0.6183605	best: 0.6183605 (0)	total: 24.9ms	remaining: 17.1s
100:	learn: 0.1536551	test: 0.1730641	best: 0.1730625 (95)	total: 1.48s	remaining: 8.58s
200:	learn: 0.1520515	test: 0.1723812	best: 0.1723811 (199)	total: 2.58s	remaining: 6.22s
300:	learn: 0.1507802	test: 0.1719313	best: 0.1719137 (297)	total: 3.95s	remaining: 5.05s
400:	learn: 0.1497046	test: 0.1716229	best: 0.1716186 (399)	total: 5.4s	remaining: 3.84s
500:	learn: 0.1486014	test: 0.1714696	best: 0.1714672 (495)	total: 6.87s	remaining: 2.54s
600:	learn: 0.1474058	test: 0.1713703	best: 0.1713703 (600)	total: 8.52s	remaining: 1.2s
685:	learn: 0.1463684	test: 0.1712566	best: 0.1712566 (685)	total: 9.9s	remaining: 0us

bestTest = 0.1712565859
bestIteration = 685

MAE:  115.42
RMSE: 647.19
R²:   0.8207


# Run XGBoost Model

## prepare XGBoost data

In [24]:
categorical_cols = ["pickup", "delivery", "equipment"]

X_dev_xgb = pd.get_dummies(
    X_dev,
    columns=categorical_cols,
    dummy_na=True
)

X_holdout_xgb = pd.get_dummies(
    X_holdout,
    columns=categorical_cols,
    dummy_na=True
)

# Make sure both datasets have exactly the same columns
X_holdout_xgb = X_holdout_xgb.reindex(
    columns=X_dev_xgb.columns,
    fill_value=0
)

print(X_dev_xgb.shape)
print(X_holdout_xgb.shape)

(38400, 139)
(9600, 139)


In [26]:
import optuna
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def objective(trial):
    """
    Train an XGBoost regressor with trial-specific hyperparameters
    and return the validation RMSE.

    Args:
        trial: Optuna trial containing sampled hyperparameters.

    Returns:
        RMSE on the October holdout set.
    """
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.15, log=True
        ),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int(
            "min_child_weight", 1, 15
        ),
        "subsample": trial.suggest_float(
            "subsample", 0.6, 1.0
        ),
        "colsample_bytree": trial.suggest_float(
            "colsample_bytree", 0.6, 1.0
        ),
        "gamma": trial.suggest_float(
            "gamma", 0, 5
        ),
        "reg_alpha": trial.suggest_float(
            "reg_alpha", 1e-4, 10, log=True
        ),
        "reg_lambda": trial.suggest_float(
            "reg_lambda", 1e-3, 20, log=True
        ),
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "random_state": 42,
        "n_jobs": -1,
    }

    model = xgb.XGBRegressor(**params)

    model.fit(
        X_dev_xgb,
        y_dev,
        eval_set=[(X_holdout_xgb, y_holdout)],
        verbose=False
    )

    predictions = model.predict(X_holdout_xgb)

    return mean_squared_error(
        y_holdout,
        predictions
    ) ** 0.5

In [27]:
study_xgb = optuna.create_study(
    direction="minimize",
    study_name="xgboost_freight_rate"
)

study_xgb.optimize(
    objective,
    n_trials=30
)

print("Best RMSE:", study_xgb.best_value)
print("Best parameters:")
print(study_xgb.best_params)

[I 2026-09-13 08:24:20,754] A new study created in memory with name: xgboost_freight_rate
[I 2026-09-13 08:24:29,524] Trial 0 finished with value: 534.9525478392968 and parameters: {'n_estimators': 439, 'learning_rate': 0.014173062606362162, 'max_depth': 8, 'min_child_weight': 8, 'subsample': 0.6902963316601629, 'colsample_bytree': 0.743231360492056, 'gamma': 1.3427523816131337, 'reg_alpha': 0.0066290502271897905, 'reg_lambda': 11.302282163820943}. Best is trial 0 with value: 534.9525478392968.
[I 2026-09-13 08:24:50,991] Trial 1 finished with value: 555.9937097415442 and parameters: {'n_estimators': 1287, 'learning_rate': 0.01421279574648416, 'max_depth': 8, 'min_child_weight': 3, 'subsample': 0.9052966699611971, 'colsample_bytree': 0.6108759142168989, 'gamma': 0.20660543238912032, 'reg_alpha': 0.0012624190369962453, 'reg_lambda': 0.0048947566937450745}. Best is trial 0 with value: 534.9525478392968.
[I 2026-09-13 08:25:06,189] Trial 2 finished with value: 576.9386408166713 and parame

Best RMSE: 528.6541009270501
Best parameters:
{'n_estimators': 671, 'learning_rate': 0.02197308252417513, 'max_depth': 3, 'min_child_weight': 7, 'subsample': 0.6043454112550912, 'colsample_bytree': 0.7852584547811419, 'gamma': 4.717802129107983, 'reg_alpha': 0.00010449319218623953, 'reg_lambda': 19.788328089945626}


## evaluation

In [28]:
best_xgb = xgb.XGBRegressor(
    **study_xgb.best_params,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1
)

best_xgb.fit(
    X_dev_xgb,
    y_dev,
    eval_set=[(X_holdout_xgb, y_holdout)],
    verbose=False
)

pred_xgb = best_xgb.predict(X_holdout_xgb)

mae_xgb = mean_absolute_error(y_holdout, pred_xgb)
rmse_xgb = mean_squared_error(y_holdout, pred_xgb) ** 0.5
r2_xgb = r2_score(y_holdout, pred_xgb)

print(f"MAE:  {mae_xgb:.2f}")
print(f"RMSE: {rmse_xgb:.2f}")
print(f"R²:   {r2_xgb:.4f}")

MAE:  113.57
RMSE: 528.65
R²:   0.8693


# Run MLP Model

In [108]:
import torch
import torch.nn as nn

from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Preprocess the data

In [110]:
categorical_cols = ["pickup", "delivery", "equipment"]

numerical_cols = [
    "distance",
    "weight",
    "month",
    "day_of_month",
    "day_of_week"
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numerical_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ]
)

# Fit preprocessing on the training data
X_dev_mlp = preprocessor.fit_transform(X_dev)

# Apply the same transformations to the holdout set
X_holdout_mlp = preprocessor.transform(X_holdout)

X_dev_mlp = X_dev_mlp.astype(np.float32)
X_holdout_mlp = X_holdout_mlp.astype(np.float32)

y_dev_mlp = y_dev.values.astype(np.float32)
y_holdout_mlp = y_holdout.values.astype(np.float32)

print("Train shape:", X_dev_mlp.shape)
print("Holdout shape:", X_holdout_mlp.shape)

Train shape: (43147, 136)
Holdout shape: (4853, 136)


## define the MLP

In [111]:
class FreightMLP(nn.Module):
    """
    Feed-forward neural network for freight-rate regression.

    Args:
        input_dim: Number of input features.
        hidden_dims: List containing the size of each hidden layer.
        dropout: Dropout probability.

    Returns:
        Neural network producing one continuous freight-rate prediction.
    """

    def __init__(self, input_dim, hidden_dims, dropout):
        super().__init__()

        layers = []
        previous_dim = input_dim

        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            previous_dim = hidden_dim

        layers.append(nn.Linear(previous_dim, 1))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(1)

In [112]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def objective_mlp(trial):
    """
    Train one MLP configuration and return its holdout RMSE.

    Args:
        trial: Optuna trial containing sampled hyperparameters.

    Returns:
        RMSE on the October holdout set.
    """

    n_layers = trial.suggest_int("n_layers", 1, 3)

    hidden_dims = []

    for i in range(n_layers):
        hidden_dims.append(
            trial.suggest_int(
                f"hidden_dim_{i}",
                32,
                512,
                step=32
            )
        )

    dropout = trial.suggest_float("dropout", 0.0, 0.5)

    learning_rate = trial.suggest_float(
        "learning_rate",
        1e-4,
        5e-3,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-6,
        1e-2,
        log=True
    )

    batch_size = trial.suggest_categorical(
        "batch_size",
        [32, 64, 128, 256]
    )

    model = FreightMLP(
        input_dim=X_dev_mlp.shape[1],
        hidden_dims=hidden_dims,
        dropout=dropout
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    criterion = nn.MSELoss()

    train_dataset = TensorDataset(
        torch.tensor(X_dev_mlp),
        torch.tensor(y_dev_mlp)
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )

    X_holdout_tensor = torch.tensor(X_holdout_mlp).to(device)

    # Train for a fixed number of epochs
    model.train()

    for epoch in range(100):

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            predictions = model(batch_X)

            loss = criterion(predictions, batch_y)

            loss.backward()
            optimizer.step()

    # Evaluate
    model.eval()

    with torch.no_grad():
        predictions = model(X_holdout_tensor).cpu().numpy()

    rmse = mean_squared_error(
        y_holdout_mlp,
        predictions
    ) ** 0.5

    return rmse

In [ ]:
study_mlp = optuna.create_study(
    direction="minimize",
    study_name="mlp_freight_rate"
)

study_mlp.optimize(
    objective_mlp,
    n_trials=30
)

print("Best RMSE:", study_mlp.best_value)
print("Best parameters:")
print(study_mlp.best_params)

## evaluation

In [113]:
# Train the final MLP using the best Optuna configuration
best_mlp = FreightMLP(
    input_dim=X_dev_mlp.shape[1],
    hidden_dims=[416],
    dropout=0.26795118695377984
).to(device)

optimizer = torch.optim.AdamW(
    best_mlp.parameters(),
    lr=0.0017024246565130283,
    weight_decay=0.000928213993175589
)

criterion = nn.MSELoss()

train_dataset = TensorDataset(
    torch.tensor(X_dev_mlp),
    torch.tensor(y_dev_mlp)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

# Train
best_mlp.train()

for epoch in range(100):
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()

        predictions = best_mlp(batch_X)
        loss = criterion(predictions, batch_y)

        loss.backward()
        optimizer.step()

# Evaluate on October holdout
best_mlp.eval()

with torch.no_grad():
    pred_mlp = best_mlp(
        torch.tensor(X_holdout_mlp).to(device)
    ).cpu().numpy()

mae_mlp = mean_absolute_error(y_holdout_mlp, pred_mlp)
rmse_mlp = mean_squared_error(y_holdout_mlp, pred_mlp) ** 0.5
r2_mlp = r2_score(y_holdout_mlp, pred_mlp)

print(f"MAE:  {mae_mlp:.2f}")
print(f"RMSE: {rmse_mlp:.2f}")
print(f"R²:   {r2_mlp:.4f}")

MAE:  171.37
RMSE: 652.49
R²:   0.8178


# Analyzing high rmse

In [87]:
print(y_dev.describe())

count    43147.000000
mean      2373.410351
std       1481.686233
min         57.220000
25%       1254.220000
50%       2029.700000
75%       3332.930000
max      25533.000000
Name: posted_rate, dtype: float64


In [88]:
print(y_holdout.describe())

count     4853.000000
mean      2379.051374
std       1528.714038
min        104.380000
25%       1231.130000
50%       2035.900000
75%       3305.450000
max      19110.300000
Name: posted_rate, dtype: float64


In [89]:
print(
    y_dev.sort_values(ascending=False).head(20).to_list()
)

[25533.0, 24294.98, 24140.21, 23662.71, 23580.42, 22755.66, 22534.65, 20361.89, 20132.37, 19042.39, 18972.15, 17772.53, 17742.34, 17688.63, 17333.65, 16951.43, 16841.6, 16386.16, 16229.57, 15165.13]


In [90]:
errors = pd.DataFrame({
    "actual": y_holdout.to_numpy(),
    "predicted": pred_cat
})

errors["sq_error"] = (
    errors["actual"] - errors["predicted"]
) ** 2

top10_share = (
    errors
    .sort_values("sq_error", ascending=False)
    .head(10)["sq_error"]
    .sum()
    / errors["sq_error"].sum()
)

print(f"Top 10 errors share of total squared error: {top10_share:.4f}")

Top 10 errors share of total squared error: 0.6021


In [91]:
top = (
    errors
    .sort_values("sq_error", ascending=False)
    .head(15)
)

print(top)

        actual    predicted      sq_error
392   17514.11  3616.775723  1.931359e+08
3066  17893.17  4199.603215  1.875138e+08
4151  19110.30  6045.663048  1.706847e+08
3369  14784.38  3687.492670  1.231409e+08
3219  14403.14  4318.690562  1.016961e+08
4389  15003.55  4956.916558  1.009348e+08
1120  14425.42  4568.648385  9.715595e+07
155   13894.55  4460.072695  8.900936e+07
237   11696.44  2456.418001  8.537801e+07
4439  11596.28  2916.730625  7.533458e+07
1197  10566.25  2196.455266  7.005346e+07
739   11052.72  3002.667864  6.480334e+07
2672   9522.01  1916.032952  5.785089e+07
495    8236.26  1936.498195  3.968700e+07
3520   9305.14  3158.280500  3.778388e+07


In [92]:
print("y_dev max:", y_dev.max())
print("y_dev 99th percentile:", y_dev.quantile(0.99))
print("y_holdout max:", y_holdout.max())

y_dev max: 25533.0
y_dev 99th percentile: 5971.349000000002
y_holdout max: 19110.3


In [93]:
X_holdout_reset = X_holdout.reset_index(drop=True)
worst_idx = errors.sort_values("sq_error", ascending=False).head(15).index
print(X_holdout_reset.loc[worst_idx])

              pickup     delivery  distance equipment   weight  month  \
392        Milwaukee  Bakersfield    1893.0   Dry Van  30760.0     10   
3066        Columbia       Tucson    2023.4   Flatbed  34710.0     10   
4151          Boston  Bakersfield    2979.1    Reefer  30585.0     10   
3369          Toledo  Albuquerque    1684.3    Reefer  31536.0     10   
3219     Bakersfield     Columbia    2251.9   Dry Van  40729.0     10   
4389     Los Angeles     Richmond    2782.4   Dry Van  25730.0     10   
1120          Albany  Albuquerque    2492.6   Dry Van  25513.0     10   
155          Atlanta      Phoenix    2083.1    Reefer  40335.0     10   
237        Lexington      Lubbock    1187.0   Dry Van  25359.0     10   
4439  Corpus Christi    Las Vegas    1383.3   Flatbed  18645.0     10   
1197       Lexington       Albany    1040.0   Dry Van  36569.0     10   
739       Shreveport    Baltimore    1348.4    Reefer  25462.0     10   
2672         Atlanta    Baltimore     940.4   Dry V

# Preparing the final Model

In [94]:
print("Best iterations:", best_cat.get_best_iteration() + 1)
print("Best parameters:", study_cat.best_params)

Best iterations: 686
Best parameters: {'iterations': 686, 'learning_rate': 0.10486604635909791, 'depth': 7, 'l2_leaf_reg': 6.988990741530071, 'random_strength': 1.1828053962394336, 'bagging_temperature': 0.7804249353008069}


## Prepare the full labeled dataset

In [95]:
# Prepare the complete labeled dataset for final training
X_full = prepare_features(train)
y_full = train["posted_rate"]

# Transform the target to log-space
y_full_log = np.log1p(y_full)

# Categorical feature indices
categorical_features_full = [
    X_full.columns.get_loc("pickup"),
    X_full.columns.get_loc("delivery"),
    X_full.columns.get_loc("equipment"),
]

print("Training rows:", len(X_full))
print("Features:", X_full.columns.tolist())

Training rows: 48000
Features: ['pickup', 'delivery', 'distance', 'equipment', 'weight', 'month', 'day_of_month', 'day_of_week']


## Train the final CatBoost model

In [96]:
# Train the final CatBoost model on all labeled data
final_model = CatBoostRegressor(
    **study_cat.best_params,
    loss_function="RMSE",
    random_seed=42,
    verbose=100
)

final_model.fit(
    X_full,
    y_full_log,
    cat_features=categorical_features_full
)

0:	learn: 0.6043244	total: 24.6ms	remaining: 16.9s
100:	learn: 0.1546846	total: 2.02s	remaining: 11.7s
200:	learn: 0.1528043	total: 3.82s	remaining: 9.22s
300:	learn: 0.1513751	total: 5.61s	remaining: 7.18s
400:	learn: 0.1500597	total: 7.48s	remaining: 5.32s
500:	learn: 0.1490665	total: 9.44s	remaining: 3.48s
600:	learn: 0.1479668	total: 11.4s	remaining: 1.61s
685:	learn: 0.1471135	total: 13s	remaining: 0us


CatBoostRegressor(bagging_temperature=0.7804249353008069, depth=7, iterations=686, l2_leaf_reg=6.988990741530071, learning_rate=0.10486604635909791, loss_function='RMSE', random_seed=42, random_strength=1.1828053962394336, verbose=100)

## Predict on validation.csv

In [97]:
# Load the required prediction template
template = pd.read_csv("/kaggle/input/datasets/tolbaadel/dataset/validation_predictions_template.csv")

# Load validation data
validation = pd.read_csv("/kaggle/input/datasets/tolbaadel/dataset/validation.csv")

# Create predictions for validation.csv
X_validation = prepare_features(validation)

validation_pred_log = final_model.predict(X_validation)
validation_predictions = np.expm1(validation_pred_log)

# Map each validation load_id to its prediction
prediction_map = dict(
    zip(validation["load_id"], validation_predictions)
)

# Fill the template using its existing load_id values
template["predicted_rate"] = template["load_id"].map(prediction_map)

# Save the completed template
template.to_csv("/kaggle/working/validation_predictions.csv", index=False)

print(template.head())
print("Rows:", len(template))
print("Missing predictions:", template["predicted_rate"].isna().sum())
print("Saved: validation_predictions.csv")

     load_id  predicted_rate
0  TE-000001      821.489925
1  TE-000002     5118.203329
2  TE-000003     5058.744037
3  TE-000004     4121.076392
4  TE-000005     1697.486286
Rows: 12000
Missing predictions: 0
Saved: validation_predictions.csv


## Predict on december_chart_inputs.csv

In [98]:
december = pd.read_csv("/kaggle/input/datasets/tolbaadel/dataset/december_chart_inputs.csv")

X_december = prepare_features(december)

# Predict in log-space
december_pred_log = final_model.predict(X_december)

# Convert back to original rate scale
december["predicted_rate"] = np.expm1(december_pred_log)

december.to_csv("/kaggle/working/december_chart_inputs.csv", index=False)

print(december.head())
print("December rows:", len(december))
print("Updated: december_chart_inputs.csv")

      pickup    delivery  distance equipment  weight        date  \
0  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-01   
1  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-02   
2  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-03   
3  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-04   
4  Lexington  Fort Wayne       360   Dry Van   32000  2025-12-05   

   predicted_rate  
0      821.851121  
1      849.885241  
2      869.036325  
3      861.874986  
4      850.285995  
December rows: 31
Updated: december_chart_inputs.csv
